In [ ]:
import os

from dotenv import load_dotenv
from notion_client import Client

load_dotenv()  # take environment variables from .env.
notion = Client(auth=os.environ["NOTION_TOKEN"])

In [21]:
from typing import Any, cast

CONTAINER_BLOCK_TYPES = frozenset(
    {"column_list", "column", "synced_block", "toggle", "callout"}
)


def collect_child_databases(notion: Any, block_parent_id: str) -> list[tuple[str, str]]:
    """Collect every ``child_database`` under ``block_parent_id``, including inside columns.

    Notion nests inline databases inside ``column_list`` / ``column`` blocks. A shallow
    ``blocks.children.list`` on the page only sees top-level blocks, so databases such
    as "All Projects" are missed if the code does not recurse into these containers.
    """
    found: list[tuple[str, str]] = []
    children = cast(dict[str, Any], notion.blocks.children.list(block_parent_id)).get(
        "results", []
    )
    for block in children:
        btype = block.get("type")
        bid = block.get("id")
        if btype == "child_database":
            title = (block.get("child_database") or {}).get("title") or "Untitled"
            found.append((bid, title))
        elif btype in CONTAINER_BLOCK_TYPES:
            found.extend(collect_child_databases(notion, bid))
    return found


def nested_block_summary(notion: Any, page_id: str) -> str:
    """One-line summary of ``child_database`` and ``child_page`` blocks directly under a page."""
    labels: list[str] = []
    children = cast(dict[str, Any], notion.blocks.children.list(page_id)).get("results", [])
    for block in children:
        btype = block.get("type")
        if btype == "child_database":
            t = (block.get("child_database") or {}).get("title") or "database"
            labels.append(t)
        elif btype == "child_page":
            t = (block.get("child_page") or {}).get("title") or "page"
            labels.append(t)
    return ", ".join(labels)


def find_page_id_by_exact_title(notion: Any, exact: str) -> str | None:
    """Return page id whose title matches ``exact`` (case-insensitive), using Search."""
    resp = cast(
        dict[str, Any],
        notion.search(
            query=exact,
            filter={"property": "object", "value": "page"},
            page_size=20,
        ),
    )
    exact_l = exact.lower()
    for hit in resp.get("results", []):
        title = get_page_title(hit)
        if title and title.strip().lower() == exact_l:
            return cast(str, hit["id"])
    return None


def find_child_database_id(
    notion: Any, research_page_id: str, database_title: str
) -> str | None:
    """Return database id for a ``child_database`` with the given title under the page (recursive)."""
    want = database_title.strip().lower()
    for db_id, title in collect_child_databases(notion, research_page_id):
        if title.strip().lower() == want:
            return db_id
    return None


def find_task_page_for_project(
    notion: Any,
    tasks_database_id: str,
    project_page_id: str,
    task_title: str,
) -> str | None:
    """Return the task page id for a row with the given title and Project relation."""
    meta = cast(dict[str, Any], notion.databases.retrieve(tasks_database_id))
    sources = meta.get("data_sources") or []
    if not sources:
        return None
    ds_id = sources[0]["id"]
    filt: dict[str, Any] = {
        "and": [
            {"property": "Project", "relation": {"contains": project_page_id}},
            {"property": "Tasks", "title": {"equals": task_title}},
        ],
    }
    resp = cast(dict[str, Any], notion.data_sources.query(ds_id, filter=filt, page_size=5))
    rows = resp.get("results") or []
    if not rows:
        return None
    return cast(str, rows[0]["id"])


def list_rows_for_database(
    notion: Any,
    database_id: str,
    data_sources: list[dict[str, Any]],
) -> tuple[list[dict[str, Any]], str]:
    """Return database rows via data_sources.query, or Search when retrieve lists no data sources."""
    if data_sources:
        ds_id = data_sources[0]["id"]
        rows: list[dict[str, Any]] = []
        cursor: str | None = None
        while True:
            kwargs: dict[str, Any] = {}
            if cursor:
                kwargs["start_cursor"] = cursor
            resp = cast(dict[str, Any], notion.data_sources.query(ds_id, **kwargs))
            rows.extend(resp.get("results", []))
            cursor = resp.get("next_cursor")
            if not cursor:
                break
        return rows, "data_sources"
    rows = []
    cursor = None
    while True:
        kwargs = {"filter": {"property": "object", "value": "page"}, "page_size": 100}
        if cursor:
            kwargs["start_cursor"] = cursor
        resp = cast(dict[str, Any], notion.search(**kwargs))
        for page in resp.get("results", []):
            parent = page.get("parent") or {}
            if parent.get("database_id") == database_id:
                rows.append(page)
        cursor = resp.get("next_cursor")
        if not cursor:
            break
    return rows, "search"


def get_page_title(page):
    # Try various ways that Notion exposes page titles
    title = None
    if "title" in page and page["title"]:
        title = page["title"][0].get("plain_text")
    else:
        props = page.get("properties", {})
        for v in props.values():
            if v.get("type") == "title" and v["title"]:
                title = v["title"][0].get("plain_text")
                break
    return title

def find_page_with_title(notion: Any, search_str: str) -> dict[str, Any] | None:
    """Resolve a page by title using Search; prefers the ``query`` string for precision."""
    search_results = cast(
        dict[str, Any],
        notion.search(
            query=search_str,
            filter={"property": "object", "value": "page"},
            page_size=20,
        ),
    )
    needle = search_str.lower()
    exact: dict[str, Any] | None = None
    partial: dict[str, Any] | None = None
    for result in search_results.get("results", []):
        title = get_page_title(result)
        if not title:
            continue
        tl = title.lower()
        if tl == needle:
            exact = result
            break
        if needle in tl and partial is None:
            partial = result
    return exact or partial


def tree_view():
    research_proj_page = find_page_with_title(notion, "Research Projects")
    if not research_proj_page:
        print("Research Projects (not found)")
        return

    rp_title = get_page_title(research_proj_page)
    print(f"{rp_title}")

    page_id = research_proj_page["id"]
    databases = collect_child_databases(notion, page_id)
    if not databases:
        print("└── [No child_database blocks under this page (including columns)]")
        return

    target_name = "all projects"
    primary = [(i, t) for i, t in databases if t.strip().lower() == target_name]
    rest = [(i, t) for i, t in databases if t.strip().lower() != target_name]
    to_show = primary if primary else rest + primary

    for db_idx, (db_id, db_title) in enumerate(to_show):
        is_last_db = db_idx == len(to_show) - 1
        db_prefix = "└── " if is_last_db else "├── "
        print(f"{db_prefix}{db_title}")

        db_meta = cast(dict[str, Any], notion.databases.retrieve(db_id))
        data_sources = db_meta.get("data_sources") or []
        projects, row_source = list_rows_for_database(notion, db_id, data_sources)
        if not projects:
            proj_prefix = "    └── " if is_last_db else "│   └── "
            hint = (
                "No rows: no data_sources from API or search fallback. "
                "Confirm Connections on this database. "
                f"(tried {row_source})"
            )
            print(f"{proj_prefix}[{hint}]")
            continue

        for idx, proj in enumerate(projects):
            pname = None
            props = proj.get("properties", {})
            for v in props.values():
                if v.get("type") == "title" and v["title"]:
                    pname = v["title"][0].get("plain_text")
                    break
            pname = pname or "[Untitled project]"
            is_last_proj = idx == len(projects) - 1
            if is_last_db:
                proj_prefix = "    └── " if is_last_proj else "    ├── "
            else:
                proj_prefix = "│   └── " if is_last_proj else "│   ├── "
            nested = nested_block_summary(notion, proj["id"])
            extra = f"  -> {nested}" if nested else ""
            print(f"{proj_prefix}{pname}{extra}")

tree_view()

Research Projects
└── All Projects
    ├── ZnPc Manuscript  -> ZnPc Manuscript, DFT Optical Model, Slab Structure, Profile Structure
    ├── Dichroic Tomography  -> Tasks
    ├── Xray Atlas  -> WSU-Carbon-Lab/xray-atlas Issues
    ├── Victors Optical Model
    ├── Samsung Grant
    ├── Optical Model Development  -> Tasks
    ├── ALS API Development  -> Tasks
    ├── Energy Dispersive XRR  -> Tasks
    ├── Physics 410 Help  -> Tasks (1)
    ├── Polystyrene Side Chains  -> Tasks (1)
    ├── Beamtime  -> Tasks (1)
    └── NN - Physics  -> Tasks (1)


In [22]:
PROJECT_PAGE_TITLE = "ZnPc Manuscript"
TASKS_DATABASE_TITLE = "Tasks"


def task_title_and_schedule(props: dict[str, Any]) -> tuple[str, str, str]:
    """Extract task title, status name, and schedule string from Notion task row properties."""
    name = ""
    status_s = ""
    schedule_s = ""
    for key, val in props.items():
        t = val.get("type")
        if t == "title" and val.get("title"):
            name = val["title"][0].get("plain_text") or ""
        elif t == "status" and val.get("status"):
            status_s = (val.get("status") or {}).get("name") or ""
        elif t == "date" and val.get("date"):
            d = val["date"]
            start = d.get("start") or ""
            end = d.get("end") or ""
            schedule_s = start if not end else f"{start} -> {end}"
    return name, status_s, schedule_s


def tasks_for_project_page(notion: Any, project_page_id: str, tasks_database_id: str) -> list[dict[str, Any]]:
    """Return all task rows in the Tasks database whose ``Project`` relation contains ``project_page_id``."""
    meta = cast(dict[str, Any], notion.databases.retrieve(tasks_database_id))
    sources = meta.get("data_sources") or []
    if not sources:
        return []
    ds_id = sources[0]["id"]
    out: list[dict[str, Any]] = []
    cursor: str | None = None
    filt: dict[str, Any] = {
        "property": "Project",
        "relation": {"contains": project_page_id},
    }
    while True:
        body: dict[str, Any] = {"filter": filt, "page_size": 100}
        if cursor:
            body["start_cursor"] = cursor
        resp = cast(dict[str, Any], notion.data_sources.query(ds_id, **body))
        out.extend(resp.get("results", []))
        cursor = resp.get("next_cursor")
        if not cursor:
            break
    return out


rp = find_page_with_title(notion, "Research Projects")
if not rp:
    raise RuntimeError("Research Projects page not found")

tasks_db_id = find_child_database_id(notion, rp["id"], TASKS_DATABASE_TITLE)
if not tasks_db_id:
    raise RuntimeError(
        f'No child database titled "{TASKS_DATABASE_TITLE}" under Research Projects'
    )

project_id = find_page_id_by_exact_title(notion, PROJECT_PAGE_TITLE)
if not project_id:
    raise RuntimeError(f'Page "{PROJECT_PAGE_TITLE}" not found via Search')

task_rows = tasks_for_project_page(notion, project_id, tasks_db_id)
print(f"Tasks linked to {PROJECT_PAGE_TITLE!r} (Project relation): {len(task_rows)}\n")


def schedule_start_iso(row: dict[str, Any]) -> str:
    """Sort key: Schedule start date, or empty string if unset."""
    sched = (row.get("properties") or {}).get("Schedule")
    if not isinstance(sched, dict) or sched.get("type") != "date":
        return ""
    dt = sched.get("date") or {}
    return dt.get("start") or ""


for row in sorted(task_rows, key=schedule_start_iso):
    props = row.get("properties", {})
    title, status, sched = task_title_and_schedule(props)
    sched_part = f" | schedule: {sched}" if sched else ""
    print(f"- [{status or '?'}] {title}{sched_part}")


Tasks linked to 'ZnPc Manuscript' (Project relation): 18

- [Blocked] Figure (Optical Model Comparison)
- [Blocked] Figure (XRR Comparison for each model)
- [Stashed] Introduction | schedule: 2026-01-14 -> 2026-01-16
- [Done] Chose Titles and Author List | schedule: 2026-01-15
- [Done] SI Figure (Effect of anisotropy) | schedule: 2026-01-20 -> 2026-01-22
- [Done] Profile model using lower resolution | schedule: 2026-01-21 -> 2026-01-23
- [Done] Fit using the the profile with 20 slabs | schedule: 2026-01-27 -> 2026-02-03
- [Done] Profile model using surface | schedule: 2026-02-03 -> 2026-02-05
- [Done] SPIE Introduction Slide | schedule: 2026-02-18T09:45:00.000-08:00 -> 2026-02-18T19:00:00.000-08:00
- [Done] SPIE Fitting Results | schedule: 2026-02-19
- [Done] SPIE Simulation Results | schedule: 2026-02-20
- [Done] SPIE Slides Due | schedule: 2026-02-20T23:59:00.000-08:00 -> 2026-02-20T23:59:00.000-08:00
- [Done] Find a better way to deal with the energy offset term | schedule: 2026-02-

In [23]:
from notion_client.helpers import collect_paginated_api

PROJECT_PAGE_TITLE = "ZnPc Manuscript"
TASKS_DATABASE_TITLE = "Tasks"
TASK_ROW_TITLE = "Figure (Optical Model Comparison)"


def rich_text_to_markdown(rich_text: list[dict[str, Any]]) -> str:
    """Turn Notion ``rich_text`` into a minimal Markdown string (bold/italic/code/strike)."""
    parts: list[str] = []
    for item in rich_text or []:
        txt = item.get("plain_text") or ""
        ann = item.get("annotations") or {}
        if ann.get("code"):
            txt = f"`{txt}`"
        elif ann.get("bold"):
            txt = f"**{txt}**"
        elif ann.get("italic"):
            txt = f"*{txt}*"
        elif ann.get("strikethrough"):
            txt = f"~~{txt}~~"
        parts.append(txt)
    return "".join(parts)


def block_to_markdown_lines(block: dict[str, Any]) -> list[str]:
    """Map a single Notion block to zero or more Markdown lines."""
    btype = block.get("type")
    if not btype:
        return []
    payload = cast(dict[str, Any], block.get(btype) or {})
    rt = cast(list[dict[str, Any]], payload.get("rich_text") or [])
    text = rich_text_to_markdown(rt)
    if btype == "paragraph":
        return [text] if text.strip() else []
    if btype == "heading_1":
        return [f"# {text}".rstrip()]
    if btype == "heading_2":
        return [f"## {text}".rstrip()]
    if btype == "heading_3":
        return [f"### {text}".rstrip()]
    if btype == "bulleted_list_item":
        return [f"- {text}".rstrip()]
    if btype == "numbered_list_item":
        return [f"1. {text}".rstrip()]
    if btype == "to_do":
        checked = payload.get("checked") or False
        box = "[x]" if checked else "[ ]"
        return [f"- {box} {text}".rstrip()]
    if btype == "quote":
        return [f"> {text}".rstrip()] if text else []
    if btype == "divider":
        return ["---"]
    if btype == "code":
        lang = (payload.get("language") or "").strip()
        body = rich_text_to_markdown(rt)
        return [f"```{lang}", body, "```"] if body or lang else ["```", "```"]
    return []


def iter_blocks_depth_first(notion: Any, parent_block_id: str):
    for block in collect_paginated_api(
        notion.blocks.children.list,
        block_id=parent_block_id,
        page_size=100,
    ):
        yield block
        if block.get("has_children"):
            bid = cast(str, block.get("id"))
            yield from iter_blocks_depth_first(notion, bid)


def page_body_markdown(notion: Any, page_id: str) -> str:
    """Concatenate all blocks under ``page_id`` into Markdown (depth-first)."""
    lines: list[str] = []
    for block in iter_blocks_depth_first(notion, page_id):
        lines.extend(block_to_markdown_lines(block))
    return "\n".join(lines).strip()


rp2 = find_page_with_title(notion, "Research Projects")
if not rp2:
    raise RuntimeError("Research Projects page not found")
tasks_db2 = find_child_database_id(notion, rp2["id"], TASKS_DATABASE_TITLE)
if not tasks_db2:
    raise RuntimeError("Tasks database not found")
proj2 = find_page_id_by_exact_title(notion, PROJECT_PAGE_TITLE)
if not proj2:
    raise RuntimeError("Project page not found")

task_page = find_task_page_for_project(notion, tasks_db2, proj2, TASK_ROW_TITLE)
if not task_page:
    raise RuntimeError(f"Task {TASK_ROW_TITLE!r} not found for this project")

md = page_body_markdown(notion, task_page)
print(md)
print("---")
print(f"length: {len(md)} characters")



---
length: 0 characters


In [30]:
MANAGED_DATABASE_TITLE = "Notebook Managed (API)"
MANAGED_DATABASE_PLOT_PROPERTY = "Plot"
PROJECT_PAGE_TITLE = "ZnPc Manuscript"
TASKS_DATABASE_TITLE = "Tasks"
TASK_ROW_TITLE = "Figure (Optical Model Comparison)"


def get_database_title(db: dict[str, Any]) -> str:
    parts = db.get("title") or []
    return "".join(p.get("plain_text", "") for p in parts).strip()


def find_existing_managed_database(
    notion: Any, parent_page_id: str, title: str
) -> str | None:
    """Resolve a database id by title: child blocks under the parent page, then Search.

    Notion Search no longer accepts ``filter: {object: database}`` (only ``page`` or
    ``data_source``). We list ``child_database`` blocks under ``parent_page_id`` first,
    then paginated ``search(query=...)`` and keep hits where ``object == 'database'``.
    """
    want = title.lower()
    for db_id, db_title in collect_child_databases(notion, parent_page_id):
        if db_title.strip().lower() == want:
            return db_id
    cursor: str | None = None
    while True:
        kwargs: dict[str, Any] = {"query": title, "page_size": 100}
        if cursor:
            kwargs["start_cursor"] = cursor
        resp = cast(dict[str, Any], notion.search(**kwargs))
        for obj in resp.get("results", []):
            o = cast(dict[str, Any], obj)
            if o.get("object") != "database":
                continue
            if get_database_title(o).lower() != want:
                continue
            bid = cast(str, o["id"])
            meta = cast(dict[str, Any], notion.databases.retrieve(bid))
            par = meta.get("parent") or {}
            if par.get("type") == "page_id" and par.get("page_id") == parent_page_id:
                return bid
        cursor = resp.get("next_cursor")
        if not cursor:
            break
    return None


def ensure_managed_database(notion: Any, parent_page_id: str, title: str) -> str:
    """Create the managed database under ``parent_page_id`` or reuse an existing one with the same title."""
    existing = find_existing_managed_database(notion, parent_page_id, title)
    if existing:
        print(f"Using existing database {title!r} -> {existing}")
        return existing
    created = cast(
        dict[str, Any],
        notion.databases.create(
            parent={"type": "page_id", "page_id": parent_page_id},
            title=[{"type": "text", "text": {"content": title}}],
            is_inline=True,
            initial_data_source={
                "properties": {
                    "Name": {"title": {}},
                    "Note": {"rich_text": {}},
                    MANAGED_DATABASE_PLOT_PROPERTY: {"files": {}},
                }
            },
        ),
    )
    db_id = cast(str, created["id"])
    print(f"Created database {title!r} -> {db_id}")
    return db_id


def ensure_plot_files_property(
    notion: Any,
    data_source_id: str,
    property_name: str = MANAGED_DATABASE_PLOT_PROPERTY,
) -> None:
    """Ensure the data source has a ``files`` column ``property_name`` for PNG figures.

    Parameters
    ----------
    notion
        Notion API client.
    data_source_id
        Primary data source id for the managed database.
    property_name
        Files property name; must match the key used in ``pages.create`` for rows.

    Notes
    -----
    Idempotent: returns immediately when ``property_name`` already exists.
    Uses ``data_sources.update`` because schema changes are applied on the data source.
    """
    ds = cast(dict[str, Any], notion.data_sources.retrieve(data_source_id))
    props = cast(dict[str, Any], ds.get("properties") or {})
    if property_name in props:
        return
    cast(
        dict[str, Any],
        notion.data_sources.update(
            data_source_id,
            properties={property_name: {"files": {}}},
        ),
    )


rp_db = find_page_with_title(notion, "Research Projects")
if not rp_db:
    raise RuntimeError("Research Projects page not found")
tasks_db_managed = find_child_database_id(notion, rp_db["id"], TASKS_DATABASE_TITLE)
if not tasks_db_managed:
    raise RuntimeError("Tasks database not found under Research Projects")
proj_page_managed = find_page_id_by_exact_title(notion, PROJECT_PAGE_TITLE)
if not proj_page_managed:
    raise RuntimeError(f"Project page {PROJECT_PAGE_TITLE!r} not found")
task_page_for_db = find_task_page_for_project(
    notion, tasks_db_managed, proj_page_managed, TASK_ROW_TITLE
)
if not task_page_for_db:
    raise RuntimeError(
        f"Task {TASK_ROW_TITLE!r} not found for project {PROJECT_PAGE_TITLE!r}"
    )

MANAGED_DATABASE_ID = ensure_managed_database(
    notion, task_page_for_db, MANAGED_DATABASE_TITLE
)
_meta_ds = cast(dict[str, Any], notion.databases.retrieve(MANAGED_DATABASE_ID))
_ds_ids = cast(list[dict[str, Any]], _meta_ds.get("data_sources") or [])
if _ds_ids:
    ensure_plot_files_property(notion, cast(str, _ds_ids[0]["id"]))
print(
    f"Managed database parent: task {TASK_ROW_TITLE!r} on {PROJECT_PAGE_TITLE!r} -> {task_page_for_db}"
)


Created database 'Notebook Managed (API)' -> a1a5364e-19ba-4419-b09a-6a60fe99eeb2
Managed database parent: task 'Figure (Optical Model Comparison)' on 'ZnPc Manuscript' -> 20694372-421b-8011-918a-eecaf4297e4f


In [31]:
try:
    _mid = MANAGED_DATABASE_ID
except NameError as exc:
    raise RuntimeError("Run the cell above first to create or resolve MANAGED_DATABASE_ID") from exc

_meta = cast(dict[str, Any], notion.databases.retrieve(MANAGED_DATABASE_ID))
_t = get_database_title(_meta)
_ds = _meta.get("data_sources") or []
print(f"OK: retrieved database {_t!r}")
print(f"  id: {MANAGED_DATABASE_ID}")
print(f"  parent: {_meta.get('parent')}")
print(f"  data_sources ({len(_ds)}): {_ds}")
if not _ds:
    raise RuntimeError("expected at least one data source on the managed database")


OK: retrieved database 'Notebook Managed (API)'
  id: a1a5364e-19ba-4419-b09a-6a60fe99eeb2
  parent: {'type': 'page_id', 'page_id': '20694372-421b-8011-918a-eecaf4297e4f'}
  data_sources (1): [{'id': '795467b4-c7c0-4491-b533-286d84816e5d', 'name': 'Notebook Managed (API)'}]


In [ ]:
pass


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Generate some random data
x = np.linspace(0, 10, 100)
y = np.sin(x) + np.random.normal(0, 0.1, 100)

plt.figure(figsize=(10, 5))
plt.scatter(x, y, label='Random Data')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.title('Scatter Plot Example')
plt.legend()

In [34]:
import base64
from datetime import UTC, datetime
from hashlib import sha256
from pathlib import Path
from typing import Any, cast

import nbformat
from nbconvert import MarkdownExporter

NOTEBOOK_EXPORT_HEADING = "Notebook export (notion-connection.ipynb)"
RT_MAX = 1900

_IMAGE_ORDER = (
    "image/png",
    "image/jpeg",
    "image/gif",
    "image/webp",
    "image/svg+xml",
)

_MIME_EXT = {
    "image/png": ".png",
    "image/jpeg": ".jpg",
    "image/gif": ".gif",
    "image/webp": ".webp",
    "image/svg+xml": ".svg",
}


def _resolve_notebook_path() -> Path:
    cwd = Path.cwd()
    for base in (cwd, cwd.parent, cwd.parent.parent):
        p = base / "notebooks" / "notion-connection.ipynb"
        if p.is_file():
            return p.resolve()
    p = cwd / "notion-connection.ipynb"
    if p.is_file():
        return p.resolve()
    raise FileNotFoundError(
        "Could not find notebooks/notion-connection.ipynb; run from repo root or notebooks/"
    )


def _plain_from_heading(block: dict[str, Any]) -> str:
    if block.get("type") != "heading_2":
        return ""
    h2 = cast(dict[str, Any], block.get("heading_2") or {})
    parts = h2.get("rich_text") or []
    return "".join(
        cast(str, p.get("plain_text") or "") for p in cast(list[dict[str, Any]], parts)
    ).strip()


def _first_top_level_child_database_index(children: list[dict[str, Any]]) -> int | None:
    for i, b in enumerate(children):
        if b.get("type") == "child_database":
            return i
    return None


def _delete_export_before_database(
    notion: Any, page_id: str, heading_exact: str, first_db_idx: int | None
) -> None:
    children = cast(
        list[dict[str, Any]],
        collect_paginated_api(notion.blocks.children.list, block_id=page_id, page_size=100),
    )
    if first_db_idx is None:
        first_db_idx = len(children)
    for i, block in enumerate(children):
        if i >= first_db_idx:
            break
        if _plain_from_heading(block) == heading_exact:
            for j in range(first_db_idx - 1, i - 1, -1):
                bid = cast(str, children[j]["id"])
                notion.blocks.delete(bid)
            return


def _rich_text_chunks(text: str) -> list[dict[str, Any]]:
    chunks: list[dict[str, Any]] = []
    i = 0
    while i < len(text):
        part = text[i : i + RT_MAX]
        chunks.append({"type": "text", "text": {"content": part}})
        i += RT_MAX
    return chunks


def _blocks_from_code(language: str, source: str) -> list[dict[str, Any]]:
    segs = _rich_text_chunks(source)
    max_segs = 90
    out: list[dict[str, Any]] = []
    for i in range(0, len(segs), max_segs):
        out.append(
            {
                "type": "code",
                "code": {"language": language, "rich_text": segs[i : i + max_segs]},
            }
        )
    return out


def _blocks_from_text_paragraphs(text: str) -> list[dict[str, Any]]:
    segs = _rich_text_chunks(text)
    max_segs = 90
    out: list[dict[str, Any]] = []
    for i in range(0, len(segs), max_segs):
        out.append({"type": "paragraph", "paragraph": {"rich_text": segs[i : i + max_segs]}})
    return out


def _code_cell_language(cell: Any) -> str:
    meta = cell.metadata or {}
    vs = meta.get("vscode")
    if isinstance(vs, dict) and isinstance(vs.get("languageId"), str):
        lid = vs["languageId"].strip().lower()
        if lid:
            return lid
    if meta.get("pygments_lexer") == "ipython3":
        return "python"
    return "python"


def _decode_mime_binary(mime: str, raw: Any) -> bytes | None:
    if raw is None:
        return None
    if mime == "image/svg+xml" and isinstance(raw, str):
        return raw.encode("utf-8")
    if isinstance(raw, str):
        return base64.b64decode(raw, validate=False)
    if isinstance(raw, (bytes, bytearray, memoryview)):
        return bytes(raw)
    if isinstance(raw, list):
        return base64.b64decode("".join(raw), validate=False)
    return None


def _upload_image_block(notion: Any, raw: bytes, mime: str, filename: str) -> dict[str, Any]:
    created = cast(
        dict[str, Any],
        notion.file_uploads.create(filename=filename, content_type=mime),
    )
    uid = cast(str, created["id"])
    cast(
        dict[str, Any],
        notion.file_uploads.send(uid, file=(filename, raw, mime)),
    )
    return {
        "type": "image",
        "image": {
            "type": "file_upload",
            "file_upload": {"id": uid},
        },
    }


def _outputs_to_notion_blocks(
    notion: Any,
    outputs: list[dict[str, Any]],
    cell_index: int,
    img_counter: list[int],
) -> list[dict[str, Any]]:
    blocks: list[dict[str, Any]] = []
    for o in outputs:
        ot = o.get("output_type")
        if ot == "stream":
            t = o.get("text", "")
            text = t if isinstance(t, str) else "".join(t)
            if text.strip():
                blocks.extend(_blocks_from_text_paragraphs(text))
        elif ot == "error":
            tb = o.get("traceback")
            if isinstance(tb, list):
                blocks.extend(_blocks_from_text_paragraphs("\n".join(tb)))
        elif ot in ("display_data", "execute_result"):
            data = cast(dict[str, Any], o.get("data") or {})
            for mime in _IMAGE_ORDER:
                if mime not in data:
                    continue
                b = _decode_mime_binary(mime, data[mime])
                if not b:
                    continue
                img_counter[0] += 1
                ext = _MIME_EXT.get(mime, ".png")
                fname = f"cell{cell_index}-out{img_counter[0]}{ext}"
                try:
                    blocks.append(_upload_image_block(notion, b, mime, fname))
                except Exception as exc:
                    blocks.extend(
                        _blocks_from_text_paragraphs(
                            f"(image upload failed: {type(exc).__name__}: {exc})"
                        )
                    )
            plain = data.get("text/plain")
            if plain is not None:
                t = plain if isinstance(plain, str) else "".join(plain)
                if t.strip():
                    blocks.extend(_blocks_from_text_paragraphs(t))
    return blocks


def _notion_blocks_from_notebook(notion: Any, nb: nbformat.NotebookNode) -> list[dict[str, Any]]:
    out: list[dict[str, Any]] = []
    img_counter = [0]
    for ci, cell in enumerate(nb.cells):
        if cell.cell_type == "markdown":
            src = cell.source if isinstance(cell.source, str) else "".join(cell.source)
            src = src.strip()
            if not src:
                continue
            out.extend(_blocks_from_code("markdown", src))
        elif cell.cell_type == "code":
            src = cell.source if isinstance(cell.source, str) else "".join(cell.source)
            if src.strip():
                lang = _code_cell_language(cell)
                out.extend(_blocks_from_code(lang, src))
            out.extend(_outputs_to_notion_blocks(notion, cell.get("outputs", []), ci, img_counter))
        elif cell.cell_type == "raw":
            src = cell.source if isinstance(cell.source, str) else "".join(cell.source)
            src = src.strip()
            if src:
                out.extend(_blocks_from_text_paragraphs(src))
    return out


def _export_children_payload(
    notion: Any,
    notebook_path: Path,
    nb_node: nbformat.NotebookNode,
    md_export: str,
    exported_at: str,
) -> list[dict[str, Any]]:
    meta = (
        f"Source: {notebook_path}\n"
        f"Exported (UTC): {exported_at}\n"
        f"Format: nbconvert MarkdownExporter (hash); cells -> Notion blocks (code, text, images)\n"
        f"SHA256 (markdown export): {sha256(md_export.encode('utf-8')).hexdigest()}"
    )
    head: list[dict[str, Any]] = [
        {
            "type": "heading_2",
            "heading_2": {
                "rich_text": [{"type": "text", "text": {"content": NOTEBOOK_EXPORT_HEADING}}],
            },
        },
        {
            "type": "paragraph",
            "paragraph": {
                "rich_text": _rich_text_chunks(meta),
            },
        },
        {"type": "divider", "divider": {}},
    ]
    head.extend(_notion_blocks_from_notebook(notion, nb_node))
    return head


try:
    _tp = task_page_for_db
except NameError as exc:
    raise RuntimeError("Run the managed-database cell so task_page_for_db is defined") from exc

notebook_path = _resolve_notebook_path()
nb_node = nbformat.read(notebook_path.open(encoding="utf-8"), as_version=4)
md_export, _ = MarkdownExporter().from_notebook_node(nb_node)
exported_at = datetime.now(UTC).replace(microsecond=0).isoformat()

_top = cast(
    list[dict[str, Any]],
    collect_paginated_api(notion.blocks.children.list, block_id=_tp, page_size=100),
)
_first_db = _first_top_level_child_database_index(_top)

_delete_export_before_database(notion, _tp, NOTEBOOK_EXPORT_HEADING, _first_db)

children_payload = _export_children_payload(notion, notebook_path, nb_node, md_export, exported_at)
_bs = 100
_after: str | None = None
for _off in range(0, len(children_payload), _bs):
    _batch = children_payload[_off : _off + _bs]
    if _off == 0:
        _resp = cast(
            dict[str, Any],
            notion.blocks.children.append(_tp, children=_batch, position={"type": "start"}),
        )
    else:
        assert _after is not None
        _resp = cast(
            dict[str, Any],
            notion.blocks.children.append(
                _tp,
                children=_batch,
                position={"type": "after_block", "after_block": {"id": _after}},
            ),
        )
    _results = cast(list[dict[str, Any]], _resp.get("results") or [])
    if _results:
        _after = cast(str, _results[-1]["id"])
print(f"Prepended notebook export ({len(md_export)} chars markdown via nbconvert) to task page {_tp}")


Prepended notebook export (32892 chars markdown via nbconvert) to task page 20694372-421b-8011-918a-eecaf4297e4f


In [40]:
import io
from typing import Any, cast

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.figure import Figure


def _upload_png_bytes(notion: Any, png: bytes, filename: str) -> str:
    created = cast(
        dict[str, Any],
        notion.file_uploads.create(filename=filename, content_type="image/png"),
    )
    uid = cast(str, created["id"])
    cast(
        dict[str, Any],
        notion.file_uploads.send(uid, file=(filename, png, "image/png")),
    )
    return uid


def ensure_plot_files_property(
    notion: Any,
    data_source_id: str,
    property_name: str = "Plot",
) -> None:
    """Ensure the data source has a ``files`` column ``property_name`` for PNG figures.

    Parameters
    ----------
    notion
        Notion API client.
    data_source_id
        Primary data source id for the managed database.
    property_name
        Files property name; must match the key used in ``pages.create`` for rows.

    Notes
    -----
    Idempotent: returns immediately when ``property_name`` already exists.
    Uses ``data_sources.update`` because schema changes are applied on the data source.
    """
    ds = cast(dict[str, Any], notion.data_sources.retrieve(data_source_id))
    props = cast(dict[str, Any], ds.get("properties") or {})
    if property_name in props:
        return
    cast(
        dict[str, Any],
        notion.data_sources.update(
            data_source_id,
            properties={property_name: {"files": {}}},
        ),
    )


def record_figure_plot_history(
    notion: Any,
    figure: Figure,
    source_code: str,
    *,
    data_source_id: str,
    database_row_name: str,
    dpi: int = 150,
    png_filename: str = "figure.png",
    plot_property_name: str | None = None,
) -> str:
    """Record a figure and its generating code as a new row in the managed database only.

    Renders ``figure`` to PNG, uploads once, and sets row properties ``Name``, ``Note`` (code),
    and the files column (``Plot`` by default). Does not append blocks or markdown to any task
    page.

    Parameters
    ----------
    notion
        Notion API client.
    figure
        Matplotlib figure to rasterize.
    source_code
        Python source stored in the ``Note`` property (truncated to Notion limits).
    data_source_id
        Managed database data source id.
    database_row_name
        Title for the new row (``Name``).
    dpi
        Rasterization resolution for the PNG.
    png_filename
        Filename label for the uploaded image.
    plot_property_name
        Files property name; defaults to ``MANAGED_DATABASE_PLOT_PROPERTY`` if set in the
        managed-database cell, otherwise ``\"Plot\"``.

    Returns
    -------
    str
        The new row page id.

    Raises
    ------
    RuntimeError
        If ``pages.create`` does not return a page id.
    """
    buf = io.BytesIO()
    figure.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
    png = buf.getvalue()
    upload_id = _upload_png_bytes(notion, png, png_filename)
    _pname = (
        plot_property_name
        if plot_property_name is not None
        else globals().get("MANAGED_DATABASE_PLOT_PROPERTY", "Plot")
    )
    ensure_plot_files_property(notion, data_source_id, _pname)
    note_payload = source_code[:2000]
    created_row = cast(
        dict[str, Any],
        notion.pages.create(
            parent={"type": "data_source_id", "data_source_id": data_source_id},
            properties={
                "Name": {
                    "title": [
                        {"type": "text", "text": {"content": database_row_name[:2000]}}
                    ],
                },
                "Note": {
                    "rich_text": [
                        {"type": "text", "text": {"content": note_payload}}
                    ],
                },
                _pname: {
                    "files": [
                        {
                            "type": "file_upload",
                            "file_upload": {"id": upload_id},
                            "name": png_filename[:2000],
                        }
                    ]
                },
            },
        ),
    )
    row_page_id = created_row.get("id")
    if not row_page_id:
        raise RuntimeError("pages.create did not return a page id")
    return cast(str, row_page_id)


DEMO_PLOT_SOURCE = """import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 10, 100)
y = np.cos(x) + np.random.normal(0, 0.08, 100)

plt.figure(figsize=(10, 5))
plt.scatter(x, y, label="Cosine noise")
plt.xlabel("X-axis")
plt.ylabel("Y-axis")
plt.title("Cosine scatter (notebook export demo)")
plt.legend()
"""

x = np.linspace(0, 10, 100)
y = np.cos(x) + np.random.normal(0, 0.08, 100)

plt.figure(figsize=(10, 5))
plt.scatter(x, y, label="Cosine noise")
plt.xlabel("X-axis")
plt.ylabel("Y-axis")
plt.title("Cosine scatter (notebook export demo)")
plt.legend()

demo_fig = plt.gcf()

try:
    _managed_db = MANAGED_DATABASE_ID
except NameError as exc:
    raise RuntimeError("Run the managed-database cell so MANAGED_DATABASE_ID exists") from exc

_meta_m = cast(dict[str, Any], notion.databases.retrieve(_managed_db))
_ds_m = cast(list[dict[str, Any]], _meta_m.get("data_sources") or [])
_managed_ds_id: str | None = cast(str, _ds_m[0]["id"]) if _ds_m else None
if not _managed_ds_id:
    raise RuntimeError("Managed database has no data source")

_row = record_figure_plot_history(
    notion,
    demo_fig,
    DEMO_PLOT_SOURCE,
    data_source_id=_managed_ds_id,
    database_row_name="Cosine scatter (notebook export demo)",
    dpi=150,
    png_filename="cosine-scatter-notebook-demo.png",
)
print(f"Recorded plot history in managed database row {_row} (task page unchanged)")
plt.close(demo_fig)


Recorded plot history in managed database row 33994372-421b-8157-ad2f-e1a8ca41b298 (task page unchanged)
